In [6]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.row_rebuilder import rebuild_consumed_messages_to_observations


In [7]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME = str("synthetic_sensor_messages_consumed_stage")
REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")


REBUILD_STATUS = str("pending")
NUMBER_OF_SENSORS = int(52)

COMPLETE_ONLY_FLAG = True
MARK_SOURCE_REBUILT_FLAG = True

OBSERVATION_WINDOW_SIZE = int(2500)

In [8]:

engine = get_engine_from_env()


In [9]:


rebuild_result = rebuild_consumed_messages_to_observations(
    engine=engine,
    schema=SCHEMA,
    source_table=REBUILD_CONSUMED_MESSAGES_SOURCE_TABLE_NAME,
    target_table=REBUILT_CONSUMED_MESSAGES_DESTINATION_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    rebuild_status=REBUILD_STATUS,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    mark_source_rebuilt=MARK_SOURCE_REBUILT_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
)


[obs-window] 1 | observation_index 1 to 1,108


In [10]:

rebuild_result

{'status': 'rebuilt',
 'consumed_rows': 57616,
 'deduped_rows': 57616,
 'rebuilt_rows': 1108,
 'rebuilt_observations': 1108,
 'updated_source_observation_groups': 1108,
 'target_table': 'synthetic_sensor_observations_rebuilt_stage'}

----

In [11]:
validation_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        COUNT(*) AS rebuilt_row_count,
        COUNT(*) FILTER (WHERE rebuild_is_complete = TRUE) AS complete_row_count,
        MIN(observation_index) AS min_observation_index,
        MAX(observation_index) AS max_observation_index,
        MIN(observation_timestamp) AS min_observation_timestamp,
        MAX(observation_timestamp) AS max_observation_timestamp
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    """
)


In [12]:

display(validation_dataframe)

,rebuilt_row_count,complete_row_count,min_observation_index,max_observation_index,min_observation_timestamp,max_observation_timestamp
0,750000,750000,1,750000,2026-03-19 08:00:00+00:00,2026-03-28 00:19:59+00:00


---

In [13]:
sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        observation_timestamp,
        stream_state,
        phase,
        meta_episode_id,
        meta_primary_fault_type,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        rebuild_sensor_count,
        rebuild_is_complete
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    ORDER BY observation_index
    LIMIT 10
    """
)


In [14]:

display(sample_dataframe)

,dataset_id,run_id,asset_id,observation_index,observation_timestamp,stream_state,phase,meta_episode_id,meta_primary_fault_type,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,rebuild_sensor_count,rebuild_is_complete
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,1,2026-03-19 08:00:00+00:00,normal,normal,0,step_shift,2.339929,48.423782,52.319363,45.037003,609.487671,52,True
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,2,2026-03-19 08:00:01+00:00,normal,normal,0,step_shift,2.366877,48.430668,51.977501,44.732082,615.093567,52,True
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,3,2026-03-19 08:00:02+00:00,normal,normal,0,step_shift,2.418399,48.400005,52.017746,44.741520,625.548401,52,True
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,4,2026-03-19 08:00:03+00:00,normal,normal,0,step_shift,2.419097,48.411598,52.030724,44.729973,625.465088,52,True
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,5,2026-03-19 08:00:04+00:00,normal,normal,0,step_shift,2.415245,48.390038,51.796700,44.507263,625.718994,52,True
5,pump_synthetic_v1,premelt_run_001,pump_asset_001,6,2026-03-19 08:00:05+00:00,normal,normal,0,step_shift,2.407531,48.400581,51.719734,44.428181,624.135315,52,True
6,pump_synthetic_v1,premelt_run_001,pump_asset_001,7,2026-03-19 08:00:06+00:00,normal,normal,0,step_shift,2.398601,48.378510,51.699894,44.434540,621.922119,52,True
7,pump_synthetic_v1,premelt_run_001,pump_asset_001,8,2026-03-19 08:00:07+00:00,normal,normal,0,step_shift,2.398482,48.363518,51.926392,44.639126,620.877197,52,True
8,pump_synthetic_v1,premelt_run_001,pump_asset_001,9,2026-03-19 08:00:08+00:00,normal,normal,0,step_shift,2.420591,48.354713,51.802505,44.497536,625.323914,52,True
9,pump_synthetic_v1,premelt_run_001,pump_asset_001,10,2026-03-19 08:00:09+00:00,normal,normal,0,step_shift,2.420582,48.394501,52.303089,44.911942,625.518494,52,True


----

In [15]:
incomplete_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        dataset_id,
        run_id,
        asset_id,
        observation_index,
        rebuild_sensor_count,
        rebuild_is_complete,
        rebuild_notes
    FROM capstone.synthetic_sensor_observations_rebuilt_stage
    WHERE rebuild_is_complete = FALSE
    ORDER BY observation_index
    LIMIT 25
    """
)

display(incomplete_dataframe)

,dataset_id,run_id,asset_id,observation_index,rebuild_sensor_count,rebuild_is_complete,rebuild_notes


----